In [ ]:
import os
import re
import sys
import math
import numpy as np
from numpy.linalg import inv
from scipy.signal import lti, lsim
from pathlib import Path
from tqdm.notebook import tqdm

if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from filter_OU_inputs import fit_TF, filter_inputs
import multitone
from multitone import newman_phase, compute_fourier_coeffs

In [ ]:
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
fontsize = 9
FIGURES_DIR = Path('figures')

In [ ]:
def rad2deg(x_rad, ref_x_deg):
    ref_y = np.asarray(ref_x_deg)
    y = np.rad2deg(np.asarray(x_rad))
    if np.any(y - ref_y > 180):
        return y - 360
    if np.any(y - ref_y < -180):
        return y + 360
    return y

In [ ]:
def find_full_var_names(TF_var_names, device_names, device_types, var_names):
    full_var_names = []
    for dev_name, dev_type in zip(device_names, device_types):
        for var_name in var_names:
            full_var_names += [
                name for name in TF_var_names if
                    re.match(f'.*-{dev_name}\\.{dev_type}\\.{var_name}$', name) is not None
            ]
    return full_var_names

In [ ]:
# input_name = 'General Load'
# input_name = 'DSL Model'
# input_name = 'pq_buffer'
# input_name = 'EqX_CA4CDI1501TRV_____LOAD____'
input_name = 'Genstat_01'
base_data_dir = Path('../data')
if input_name == 'General Load':
    network_name = 'Simple_Grid'
    system_frequency = 50
    ref_SM_name = 'SG_01'
    data_dir = base_data_dir / network_name / 'without_buffer'
    fmin, fmax = -6, 2
    points_per_decade = 1000
    devices_to_consider = 'gen', 'bus'
    N_tones, F0 = 20, 50e-3
    dP, dt = 1e-3, None
    input_names = [input_name]
elif input_name == 'DSL Model':
    network_name = 'Simple_Grid'
    system_frequency = 50
    ref_SM_name = 'SG_01'
    data_dir = base_data_dir / network_name / 'with_buffer'
    fmin, fmax = -6, 2
    points_per_decade = 1000
    devices_to_consider = 'gen', 'bus'
    dP, dt = 1e-3, 5e-4
    N_tones, F0 = 50, 0.01
    N_tones, F0 = 50, 0.1
    input_names = [input_name]
elif input_name == 'pq_buffer':
    network_name = 'Nine_Bus_System'
    system_frequency = 60
    ref_SM_name = 'G1'
    condition =  'without_GFM'
    data_dir = base_data_dir / network_name / condition
    if condition == 'with_GFM':
        Ta = 4
        data_dir /= 'Ta_{}'.format(Ta)
    fmin, fmax = -6, 2
    points_per_decade = 1000
    devices_to_consider = 'gen', 'bus'
    dP = 1e-3
    N_tones, F0 = 50, 0.01
    N_tones, F0 = 50, 0.1
    dt = 5e-4
    input_names = [input_name]
elif input_name == 'EqX_CA4CDI1501TRV_____LOAD____':
    # suffix = '_lowLV'
    suffix = ''
    Ta = 4
    network_name = 'Sardegna_2021_06_03cr_1GFM' + suffix
    data_dir = base_data_dir / 'Sardinia' / 'default_config' / 'GFM_01{}/{}/Ta_{}'.format(suffix, input_name, Ta)
    fmin, fmax = -6, 2
    points_per_decade = 50
    devices_to_consider = 'gen',
    dP = 1e-3
    N_tones, F0 = 20, 50e-3
    N_tones, F0 = 40, 5e-3
    dt = 5e-3
    N_tones, F0 = 50, 0.01
    # N_tones, F0 = 50, 0.1
    dt = 5e-4
    input_names = ['pq_buffer']
elif input_name == 'Genstat_01':
    network_name = 'Sardegna_2021_06_03cr_multiple_GFM'
    ref_SM_name = 'CODCTI0201GGR1____GEN_____'
    system_frequency = 50
    Ta = 10
    data_dir = base_data_dir / 'Sardinia' / 'default_config' / 'multiple_GFM' / f'Ta_{Ta}'
    fmin, fmax = -6, 2
    points_per_decade = 100
    devices_to_consider = 'gen', 'bus'
    dP = 1e-3
    dt = 5e-4
    N_tones, F0 = 50, 0.1
    input_names = [input_name]
else:
    raise Exception(f"Do not know input name '{input_name}'")
assert data_dir.exists(), f'{data_dir}: no such directory.'

Load the AC data: the data file contains the transfer functions computed by `compute_spectra.py`.

In [ ]:
AC_data_file = data_dir / ('{}_AC_TF_{:.1f}_{:.1f}_{}.npz'.format(network_name, fmin, fmax, points_per_decade))
assert AC_data_file.exists(), f'{AC_data_file}: no such file'
AC_data = np.load(AC_data_file, allow_pickle=True)
frequencies = AC_data['F']
TF_var_names = AC_data['var_names'].tolist()

In [ ]:
names = []
idx = []
for dev_name, var_names in AC_data['vars_idx'].item().items():
    for n, i in var_names.items():
        assert len(i) == 1
        names.append(dev_name + '.' + n)
        idx.append(i[0])
all_var_names = [names[i] for i in np.argsort(idx)]

Load the transient data: the data file contains the results of the simulations performed with PowerFactory.

In [ ]:
tran_data_file = data_dir / (
    '{}_tran{}_F0_{}_Hz_dP_{}{}.npz'.format(
        network_name,
        f'_multitone_N_{N_tones}' if N_tones is not None else '',
        F0,
        dP,
        f'_dt_{dt}' if dt is not None else '',
    )
)
assert tran_data_file.exists(), f'{tran_data_file}: no such file'
blob = np.load(tran_data_file, allow_pickle=True)
time_tran = blob['time'].astype(float)
tran_data = blob['data'].item()
tran_config = blob['config'].item()
input_waveform = tran_config['inputs'][0]['waveform']
print('Input waveform:', input_waveform)
input_fun = np.vectorize(eval(input_waveform), otypes=[list])
match = re.search('\\[[A-Za-z0-9\\. =+\\-\\*]*\\(', input_waveform)
assert match is not None, 'RegEx does not match input waveform'
substr = match.group()[1:]
print(substr)
if 'else' in substr:
    substr = substr.split(' else ')[1]
tran_coeff = 1
for pos in re.finditer('[+-]', substr):
    tran_coeff = float(substr[:pos.span()[0]])
    substr = substr[pos.span()[-1]:]
substr = substr.strip()
for pos in re.finditer('[+*/\\-]', substr):
    stop = pos.span()[1] - 1
if tran_coeff == 0:
    tran_coeff = 1
print('input_coeff = 1 / ({})'.format(substr[:stop]))
input_coeff = 1 / eval(substr[:stop])
print('Input coefficient:', input_coeff)
print('Tran coefficient:', tran_coeff)
device_names = blob['device_names'].item()
device_types_dict = {'genstat': 'ElmGenstat', 'gen': 'ElmSym', 'bus': 'ElmTerm'}
var_names_dict = {
    'genstat': ['s:xspeed'],
    'gen': ['s:speed'],
    'bus': ['m:ur', 'm:ui', 'm:fe'],
    # 'bus': ['m:ur', 'm:ui', 'm:u', 'm:u2', 'm:delta', 'm:fe'],
}
tran_data_dict = {
    'genstat': ['s:xspeed'],
    'gen': ['s:xspeed'],
    'bus': ['m:ur', 'm:ui', 'm:fe'],
    # 'bus': ['m:ur', 'm:ui', 'm:u', 'm:u2', 'm:delta', 'm:fe'],
}

### add the missing variables
if 'm:u2' not in tran_data['bus'] and 'm:u2' in var_names_dict['bus']:
    tran_data['bus']['m:u2'] = tran_data['bus']['m:u'] ** 2
if 'm:delta' not in tran_data['bus'] and 'm:delta' in var_names_dict['bus']:
    tran_data['bus']['m:delta'] = np.atan2(tran_data['bus']['m:ui'], tran_data['bus']['m:ur'])

##### Define the inputs and outputs among the variables available in the data file:

Inputs:

In [ ]:
all_input_names = AC_data['input_names'].tolist()
assert all(name in all_input_names for name in input_names), \
    "At least one input name missing from data file"
input_idx = np.array([all_input_names.index(name) for name in input_names])
N_inputs = len(input_idx)
print('Number of inputs:', N_inputs)
print('Input index:', *input_idx)

Outputs:

In [ ]:
full_var_names = []
N_tran_samples = tran_data['gen']['s:xspeed'].shape[0]
max_N_vars = 20
Y_tran = []
for dev in devices_to_consider:
    tmp = find_full_var_names(
        TF_var_names,
        device_names[dev][:max_N_vars],
        [device_types_dict[dev] for _ in range(max_N_vars)],
        [v.split(':')[1] for v in var_names_dict[dev]]
    )
    full_var_names += tmp
    N_vars = len(tran_data_dict[dev])
    N = len(tmp)
    Y_tran_tmp = np.zeros((N_tran_samples, N))
    for i in range(N // N_vars):
        for j, v in enumerate(tran_data_dict[dev]):
            try:
                Y_tran_tmp[:, i * N_vars + j] = tran_data[dev][v][:, i]
            except:
                Y_tran_tmp[:, i * N_vars + j] = tran_data[dev][v]
    Y_tran.append(Y_tran_tmp)
Y_tran = tran_coeff * np.concatenate(Y_tran, axis=1).T
while True:
    if time_tran[-1] == time_tran[-2]:
        time_tran = time_tran[:-1]
        Y_tran = Y_tran[:, :-1]
        print('Removing last sample from time vector.')
    else:
        break
print('Y_tran shape:', Y_tran.shape)
TF_idx = np.array([TF_var_names.index(name) for name in full_var_names])
N_outputs = len(TF_idx)
print('Number of outputs:', N_outputs)
print('TF indexes:', *TF_idx)

Define the inputs:

In [ ]:
dt = tran_config['dt']
tend = tran_config['tstop']
time = np.r_[0 : tend + dt / 2: dt]
if N_tones is not None and N_tones > 1:
    u = multitone.multitone(time, N_tones, w0=2 * np.pi * F0) / input_coeff
    u[0] = 0
else:
    u = input_fun(time)
    u = np.array([_[0] for _ in u])
print('Mean value of input:', u.mean())
U = np.tile(
    u,
    (N_inputs, 1)
)

Make sure that the time vectors for transient and small-signal simulations are the same:

In [ ]:
if np.allclose(time, time_tran):
    print('Tran and small-signal time vectors are the same.')
else:
    raise Exception('Tran and small-signal time vectors are not the same')

Get the appropriate transfer functions:

In [ ]:
jj, kk = np.meshgrid(input_idx, TF_idx, indexing='ij')
TF = AC_data['TF'][:, jj, kk]

Fit the tranfer functions using the vector fitting algorithm and create the `lsim` objects:

In [ ]:
systems, fit = fit_TF(frequencies, TF)

Filter the inputs using the `lsim` objects above:

In [ ]:
XY_vf = filter_inputs(systems, U, time)
print('Shape of XY_vf:', XY_vf.shape)

Create a large system using the `A`, `B`, `C` and `D` matrices and the `V` vector and filter the input:

In [ ]:
for key in 'ABCDV':
    exec(f"{key} = AC_data['{key}']")
N_state_vars = A.shape[0]
N_algebraic_vars = B.shape[1]
SYS = lti(A, B @ V, C, D @ V)
_, yout, xout = lsim(SYS, u, time)
XY = np.concatenate([xout, yout], axis=1).T
print('Shape of A:', A.shape)
print('Shape of B:', B.shape)
print('Shape of C:', C.shape)
print('Shape of D:', D.shape)
print('Shape of V:', V.shape)
print('Number of state variables:', N_state_vars)
print('Number of algebraic variables:', N_algebraic_vars)
print('Shape of XY:', XY.shape)

Compute additional quantities at network buses, if these are present in the data from the tran simulation:

In [ ]:
if 'bus' in devices_to_consider and 'm:u' in var_names_dict['bus']:
    print('Adding u...')
    PF = AC_data['PF'].item()
    i = 0
    while i < len(TF_var_names) - 1:
        if TF_var_names[i][-3:] == '.ur' and TF_var_names[i + 1][-3:] == '.ui':
            ur_idx = i
            ui_idx = i + 1
            i += 1
        elif TF_var_names[i][-3:] == '.ui' and TF_var_names[i + 1][-3:] == '.ur':
            ui_idx = i
            ur_idx = i + 1
            i += 1
        else:
            ur_idx = ui_idx = None
        if ur_idx is not None:
            bus_name = TF_var_names[ur_idx].split('.')[0].split('-')[-1]
            ur, ui = XY[[ur_idx, ui_idx]]
            ur_PF, ui_PF = [PF['buses'][bus_name][f'u{_}'] for _ in 'ri']
            u1 = np.sqrt((ur + ur_PF) ** 2 + (ui + ui_PF) ** 2)
            XY = np.vstack((XY, np.atleast_2d(u1)))
            new_var_name = TF_var_names[i][-3:] + '.u'
            if new_var_name not in all_var_names:
                all_var_names.append(new_var_name)
        i += 1

In [ ]:
if 'bus' in devices_to_consider and 'm:u2' in var_names_dict['bus']:
    print('Adding u2...')
    PF = AC_data['PF'].item()
    i = 0
    while i < len(TF_var_names) - 1:
        if TF_var_names[i][-3:] == '.ur' and TF_var_names[i + 1][-3:] == '.ui':
            ur_idx = i
            ui_idx = i + 1
            i += 1
        elif TF_var_names[i][-3:] == '.ui' and TF_var_names[i + 1][-3:] == '.ur':
            ui_idx = i
            ur_idx = i + 1
            i += 1
        else:
            ur_idx = ui_idx = None
        if ur_idx is not None:
            bus_name = TF_var_names[ur_idx].split('.')[0].split('-')[-1]
            ur, ui = XY[[ur_idx, ui_idx]]
            ur_PF, ui_PF = [PF['buses'][bus_name][f'u{_}'] for _ in 'ri']
            u2 = (ur + ur_PF) ** 2 + (ui + ui_PF) ** 2
            XY = np.vstack((XY, np.atleast_2d(u2)))
            new_var_name = TF_var_names[i][-3:] + '.u2'
            if new_var_name not in all_var_names:
                all_var_names.append(new_var_name)
        i += 1

In [ ]:
if 'bus' in devices_to_consider and 'm:delta' in var_names_dict['bus']:
    print('Adding delta...')
    PF = AC_data['PF'].item()
    i = 0
    while i < len(TF_var_names) - 1:
        if TF_var_names[i][-3:] == '.ur' and TF_var_names[i + 1][-3:] == '.ui':
            ur_idx = i
            ui_idx = i + 1
            i += 1
        elif TF_var_names[i][-3:] == '.ui' and TF_var_names[i + 1][-3:] == '.ur':
            ui_idx = i
            ur_idx = i + 1
            i += 1
        else:
            ur_idx = ui_idx = None
        if ur_idx is not None:
            bus_name = TF_var_names[ur_idx].split('.')[0].split('-')[-1]
            ur, ui = XY[[ur_idx, ui_idx]]
            ur_PF, ui_PF = [PF['buses'][bus_name][f'u{_}'] for _ in 'ri']
            delta = np.atan2(ui + ui_PF, ur + ur_PF)
            XY = np.vstack((XY, np.atleast_2d(delta)))
            new_var_name = TF_var_names[i][-3:] + '.delta'
            if new_var_name not in all_var_names:
                all_var_names.append(new_var_name)
        i += 1

In [ ]:
if 'bus' in devices_to_consider and 'm:fe' in var_names_dict['bus']:
    def compute_fe(t, ur, ui, f0, omega_ref):
        delta = np.atan2(ui, ur)
        omega = np.diff(delta) / np.diff(t)
        omega = np.concatenate([omega, omega[-1:]])
        omega /= 2 * np.pi * f0
        return omega + omega_ref
    print('Adding fe...')
    PF = AC_data['PF'].item()
    ref_SM_idx = [i for i, var_name in enumerate(TF_var_names) if (ref_SM_name + '.ElmSym.speed') in var_name]
    assert len(ref_SM_idx) == 1
    omega_ref = XY[ref_SM_idx[0]] + 1
    i = 0
    while i < len(TF_var_names) - 1:
        if TF_var_names[i][-3:] == '.ur' and TF_var_names[i + 1][-3:] == '.ui':
            ur_idx = i
            ui_idx = i + 1
            i += 1
        elif TF_var_names[i][-3:] == '.ui' and TF_var_names[i + 1][-3:] == '.ur':
            ui_idx = i
            ur_idx = i + 1
            i += 1
        else:
            ur_idx = ui_idx = None
        if ur_idx is not None:
            bus_name = TF_var_names[ur_idx].split('.')[0].split('-')[-1]
            ur, ui = XY[[ur_idx, ui_idx]]
            ur_PF, ui_PF = [PF['buses'][bus_name][f'u{_}'] for _ in 'ri']
            fe = compute_fe(time, ur + ur_PF, ui + ui_PF, system_frequency, omega_ref)
            XY = np.vstack((XY, np.atleast_2d(fe)))
            new_var_name = TF_var_names[i][:-3] + '.fe'
            if new_var_name not in all_var_names:
                all_var_names.append(new_var_name)
        i += 1

In [ ]:
var_idx = np.array([all_var_names.index(name) for name in full_var_names])
assert len(TF_idx) == len(var_idx)
print('Variable indexes:', *var_idx)

In [ ]:
T0 = 1 / F0
N_T = min(20, int(tend // T0) - 5)
tstart = tend - N_T * T0
IDX, = np.where(time > tstart - dt / 2)
IDX_tran, = np.where(time_tran > tstart - dt / 2)
assert np.all(IDX == IDX_tran), 'Indices in time vectors are not the same'
print(f'Simulation duration: {tend} s.')
print(f'Longest period: {T0} s.')
print(f'Number of periods that will be used to compute the Fourier coefficients: {N_T}.')
print(f'Removing the first {tstart} s.')

Compute the Fourier coefficients:

In [ ]:
if N_tones is not None:
    freqs = np.array([(i + 1) * F0 for i in range(N_tones)])
    phases = newman_phase(np.arange(N_tones) + 1, N_tones)
    A = np.sqrt(N_tones / 2)
else:
    freqs = np.array([F0])
    phases = np.array([0.0])
    A = np.array([1.0])

if len(freqs) > 1:
    unwrap = lambda x: np.unwrap(x, axis=0)
else:
    unwrap = lambda x: x

ii, jj = np.meshgrid(var_idx, IDX, indexing='ij')
coeffs = compute_fourier_coeffs(
    freqs,
    time[IDX] - time[IDX[0]],
    XY[ii, jj],
    phases,
    A,
)
C = input_coeff * np.abs(coeffs)
ϕ = unwrap(np.angle(coeffs))

coeffs_vf = compute_fourier_coeffs(
    freqs,
    time[IDX] - time[IDX[0]],
    XY_vf[:, IDX],
    phases,
    A,
)
C_vf = input_coeff * np.abs(coeffs_vf)
ϕ_vf = unwrap(np.angle(coeffs_vf))

coeffs_tran = compute_fourier_coeffs(
    freqs,
    time[IDX] - time[IDX[0]],
    Y_tran[:, IDX],
    phases,
    A,
)
C_tran = input_coeff * np.abs(coeffs_tran)
ϕ_tran = unwrap(np.angle(coeffs_tran))

Saving the coefficients for subsequent analyses:

In [ ]:
print('Shape of C_vf:', C_vf.shape)
print('Shape of C_tran:', C_tran.shape)
print('Shape of C:', C.shape)

output_data = {
    'inputs': input_names,
    'outputs': full_var_names,
    'freqs_TF': frequencies,
    'TF': TF,
    'freqs_coeffs': freqs,
    'coeffs_ss': coeffs,
    'coeffs_ss_vf': coeffs_vf,
    'coeffs_tran': coeffs_tran,
    'input_coeff': input_coeff,
    'tran_coeff': tran_coeff,
}
outfile = tran_data_file.parent / (tran_data_file.stem + '_coeffs.npz')
np.savez_compressed(outfile, **output_data)

Plot the results:

In [ ]:
JDX, = np.where(time > tend - T0 - dt / 2)
JDX_tran, = np.where(time_tran > tend - T0 - dt / 2)
assert np.all(JDX == JDX_tran)

In [ ]:
### the index of the output variable to consider ###
K = 0
lbl = full_var_names[K].split('-')[-1]
dev, _, var = full_var_names[K].split('-')[-1].split('.')

cmap = ['tab:red', 'tab:green', 'tab:blue']
fig, ax = plt.subplots(2, 1, figsize=(6,4), height_ratios=(1,3), sharex=True)

ax[0].plot(time[JDX], u[JDX], 'k', lw=1)
m = XY[var_idx[K], JDX].mean()
print('(Reconstructed) small-signal mean:', m)
ax[1].plot(time[JDX], XY[var_idx[K], JDX] - m, color=cmap[1], lw=1, label=lbl + ' (ss)')
m = XY_vf[K, JDX].mean()
print('Vector-fitting small-signal mean:', m)
ax[1].plot(time[JDX], XY_vf[K, JDX] - m, color=cmap[2], lw=1, label=lbl + ' (ss vf)')
m = Y_tran[K, JDX].mean()
print('PowerFactory mean:', m)
ax[1].plot(time[JDX], Y_tran[K, JDX] - m, ':', color=cmap[0], lw=1, label=lbl + ' (PF tran)')
ax[0].set_ylabel('U(t)')
ax[1].set_ylabel('Y(t) - E[Y(t)]')
ax[1].legend(loc='best', frameon=False, fontsize=8)
ax[1].set_xlabel('Time (s)')
sns.despine()
fig.tight_layout()
outfile = tran_data_file.parent / (tran_data_file.stem + f'_{dev}_{var}.pdf')
print(outfile)
plt.savefig(outfile)

In [ ]:
n = 0 if N_tones is None else N_tones - 1
F_target = (n + 1) * F0
try:
    i = np.where(frequencies == F_target)[0][0]
except:
    i = np.argmin(np.abs(frequencies - F_target))
    print('Frequency {} Hz not available: closest one is {:g} Hz (discrepancy {:.2f}%).'.\
        format(F_target, frequencies[i], (F_target - frequencies[i]) / F_target * 100))
j = 0
C_TF, ϕ_TF = np.abs(TF[i, j, K]), np.angle(TF[i, j, K])

phi_TF = np.rad2deg(ϕ_TF)
if phi_TF < -90:
    phi_TF += 360
phi = rad2deg(ϕ[n, K], phi_TF)
phi_vf = rad2deg(ϕ_vf[n, K], phi_TF)
phi_tran = rad2deg(ϕ_tran[n, K], phi_TF)

print('F = {:g} Hz'.format(F_target))
print('Transfer function from PF modal analysis:')
print('    abs = {:g}'.format(C_TF))
print('  phase = {:g} deg'.format(phi_TF))
print('Fourier coefficient from small-signal simulation:')
print('    abs = {:g}'.format(C[n, K]))
print('  phase = {:g} deg'.format(phi))
print('Fourier coefficient from small-signal simulation with vector fitting:')
print('    abs = {:g}'.format(C_vf[n, K]))
print('  phase = {:g} deg'.format(phi_vf))
print('Fourier coefficient from PF transient simulation:')
print('    abs = {:g}'.format(C_tran[n, K]))
print('  phase = {:g} deg'.format(phi_tran))

In [ ]:
if N_tones is not None:
    idx = (frequencies > 2/3 * F0) & (frequencies < 1.5 * N_tones * F0)
else:
    idx = (frequencies > 2/3 * F0) & (frequencies < 1.5 * F0)
C_TF = np.abs(TF[idx, j, K])
phi_TF = np.rad2deg(np.unwrap(np.angle(TF[idx, j, K])))
if N_tones is None or N_tones == 1:
    phi = rad2deg(ϕ[:, K], phi_TF)
    phi_vf = rad2deg(ϕ_vf[:, K], phi_TF)
    phi_tran = rad2deg(ϕ_tran[:, K], phi_TF)
else:
    phi = np.rad2deg(ϕ[:, K])
    phi_vf = np.rad2deg(ϕ_vf[:, K])
    phi_tran = np.rad2deg(ϕ_tran[:, K])

ms = 3
ew = 1
coeff = 1e0
fig, ax = plt.subplots(2, 1, figsize=(6,4), sharex=True)
ax[0].plot(freqs * coeff, 20 * np.log10(C_vf[:, K]), 'o', color='tab:blue', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w', label='ss vf')
ax[0].plot(freqs * coeff, 20 * np.log10(C[:, K]), 'o', color='tab:green', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w', label='ss')
ax[0].plot(freqs * coeff, 20 * np.log10(C_tran[:, K]), 'o', color='tab:red', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w', label='PF tran')
ax[0].plot(frequencies[idx] * coeff, 20 * np.log10(C_TF), 'k--', lw=1, label='TF')
ax[1].plot(freqs * coeff, phi_vf, 'o', color='tab:blue', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w')
ax[1].plot(freqs * coeff, phi, 'o', color='tab:green', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w')
ax[1].plot(freqs * coeff, phi_tran, 'o', color='tab:red', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w')
ax[1].plot(frequencies[idx] * coeff, phi_TF + 360 * 0, 'k--', lw=1)
ax[0].set_xscale('log')
ax[0].legend(loc='best', frameon=False, fontsize=8)
ax[0].set_ylabel(r'$|\hat{X}(j\omega)|$')
ax[1].set_ylabel(r'$\phi(X(j\omega))$')
ax[1].set_xlabel('Frequency [{}Hz]'.format('m' if coeff == 1e3 else ''))
sns.despine()
fig.tight_layout()
outfile = tran_data_file.parent / (tran_data_file.stem + f'_TF_{dev}_{var}.pdf')
print(outfile)
plt.savefig(outfile)

In [ ]:
if 'Sardegna' in network_name:
    Hsm = blob['Hsm'].item()
    Ssm = blob['Ssm'].item()
    SM_names = list(Hsm)
    Ssg = blob['Ssg'].item()
    SG_names = list(Ssg)
    out = {
        'DSL_params': blob['DSL_params'].item(),
        'H_SM': np.array([Hsm[n] for n in SM_names]),
        'S_SM': np.array([Ssm[n] for n in SM_names]),
        'S_SG': Ssg,
        'SM_names': SM_names,
        'static_gen_names': SG_names,
        'Etot_SM': blob['energy'].item(),
        'F': freqs,
        'TF': coeffs_tran,
        'var_names': [v.split('-')[-1] for v in full_var_names],
        'input_names': input_names,
    }
    outfile = data_dir / (tran_data_file.stem + '_TF.npz')
    print('Saving to ', outfile)
    np.savez_compressed(outfile, **out)
    from scipy.io import savemat
    for k in out:
        if out[k] is None:
            out[k] = []
        elif isinstance(out[k], Path):
            out[k] = str(out[k])
    matfile = outfile.parent / (outfile.stem + '.mat')
    print('Saving to', matfile)
    savemat(matfile, out, long_field_names=True)